In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
import numpy as np 
import pandas as pd 
import os
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import log_loss
from datetime import datetime
import matplotlib.pyplot as plt

In [7]:
education_train = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_Education_train_set.csv')
education_test = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_Education_test_set.csv')
household_train = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_HouseholdInfo_train_set.csv')
household_test = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_HouseholdInfo_test_set.csv')
subjective_poverty_train = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_SubjectivePoverty_train_set.csv')
sample_submission = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/sample_submission.csv')
merged_data = pd.read_csv('merged_data2.csv')

/var/folders/g8/vy9w_fxd6r39lbfd4qykf2j80000gn/T/ipykernel_30363/1678864971.py:7: DtypeWarning: Columns (518,519,520,522,659,662,663,664,666,670,671,672,674) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_data = pd.read_csv('merged_data2.csv')


In [8]:
target_columns = [f'subjective_poverty_{i}' for i in range(1, 11)]

In [9]:
merged_train = merged_data.dropna(subset=target_columns)
X = merged_train.drop(columns=target_columns)
y = merged_train[target_columns]
X = X.drop(columns=X.select_dtypes(include=['object']).columns)
X = X.drop(columns=['hhid'])
y_class_labels = y.values.argmax(axis=1)



In [10]:
from sklearn.model_selection import KFold
from xgboost import XGBClassifier
from sklearn.metrics import log_loss
import numpy as np

# Function for cross-validation

def cross_validate_xgboost(X, y, y_class_labels, n_splits, reg_lambda, reg_alpha):
    cv_scores_val = []  # Validation scores
    cv_scores_train = []  # Training scores

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (train_index, val_index) in enumerate(kf.split(X), 1):
        # Split data into training and validation sets
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]

        # Get class labels for training
        y_train = y_class_labels[train_index]

        # Get one-hot encoded labels for training and validation
        y_train_onehot = y.iloc[train_index].values
        y_val_onehot = y.iloc[val_index].values

        # Train XGBoost model
        xgb_model = XGBClassifier(
            objective='multi:softprob',  # Multiclass classification
            num_class=10,                # Number of classes
            eval_metric='mlogloss',      # Multiclass log loss
            random_state=42,
            reg_lambda=reg_lambda,
            reg_alpha=reg_alpha,
        )
        xgb_model.fit(X_train, y_train)

        # Predict probabilities for training and validation sets
        y_pred_proba_train = xgb_model.predict_proba(X_train)  # Training probabilities
        y_pred_proba_val = xgb_model.predict_proba(X_val)      # Validation probabilities

        # Compute multiclass log loss for training and validation
        train_loss = log_loss(y_train_onehot, y_pred_proba_train)
        val_loss = log_loss(y_val_onehot, y_pred_proba_val)

        # Append scores
        cv_scores_train.append(train_loss)
        cv_scores_val.append(val_loss)

        print(f"Fold {fold}: Training Log Loss = {train_loss:.4f}, Validation Log Loss = {val_loss:.4f}")

    avg_train_loss = np.mean(cv_scores_train)
    avg_val_loss = np.mean(cv_scores_val)

    print("\nTraining Log Loss scores for each fold:", cv_scores_train)
    print("Validation Log Loss scores for each fold:", cv_scores_val)
    print(f"Average Training Log Loss: {avg_train_loss:.4f}")
    print(f"Average Validation Log Loss: {avg_val_loss:.4f}")

    return avg_train_loss, avg_val_loss

# Grid search over reg_lambda and reg_alpha

reg_lambda_values = [1.0, 1.5, 2.0]
reg_alpha_values = [10.0, 12.0, 15.0]

best_config = None
best_val_loss = float('inf')

for reg_lambda in reg_lambda_values:
    for reg_alpha in reg_alpha_values:
        print(f"Testing reg_lambda={reg_lambda}, reg_alpha={reg_alpha}")
        avg_train_loss, avg_val_loss = cross_validate_xgboost(X, y, y_class_labels, n_splits=7, reg_lambda=reg_lambda, reg_alpha=reg_alpha)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_config = (reg_lambda, reg_alpha)

print(f"\nBest configuration: reg_lambda={best_config[0]}, reg_alpha={best_config[1]} with Validation Log Loss: {best_val_loss:.4f}")


Testing reg_lambda=1.0, reg_alpha=10.0
Fold 1: Training Log Loss = 1.3138, Validation Log Loss = 1.9013
Fold 2: Training Log Loss = 1.3276, Validation Log Loss = 1.8921
Fold 3: Training Log Loss = 1.3391, Validation Log Loss = 1.9519
Fold 4: Training Log Loss = 1.3186, Validation Log Loss = 1.9228
Fold 5: Training Log Loss = 1.3409, Validation Log Loss = 1.8882
Fold 6: Training Log Loss = 1.3440, Validation Log Loss = 1.9007
Fold 7: Training Log Loss = 1.3529, Validation Log Loss = 1.9026

Training Log Loss scores for each fold: [1.3138178296596377, 1.3276129537700208, 1.3391440180539176, 1.3185720509639014, 1.3408769655191946, 1.3440399866774755, 1.3529050120481951]
Validation Log Loss scores for each fold: [1.9013317668772507, 1.8920745837947612, 1.9519087226720433, 1.9227500094767458, 1.8882018063126667, 1.9007110163446383, 1.9026284854666016]
Average Training Log Loss: 1.3339
Average Validation Log Loss: 1.9085
Testing reg_lambda=1.0, reg_alpha=12.0
Fold 1: Training Log Loss = 1.47

KeyboardInterrupt: 

In [ ]:
merge_test = merged_data[(merged_data['source_hh'] == 'test') | (merged_data['source_edu'] == 'test')]
psu_hh_id_merge_test = merge_test['psu_hh_idcode']
merge_test = merge_test.drop(columns=['psu_hh_idcode', 'source_hh', 'birth_date_father', 'birth_date_mother', 'birth_date', 'source_edu', 'birth_date_spouse', 'hhid'])

In [ ]:
for df in [education_train, education_test]:
    if 'psu_hh_idcode' not in df.columns:
        df['psu_hh_idcode'] = df['psu'].astype(str) + "_" + df['hh'].astype(str) + "_" + df['idcode'].astype(str)
        df.drop(columns=['psu', 'hh', 'idcode'], inplace=True)  # Remove individual columns after creating psu_hh_idcode

for df in [household_train, household_test]:
    if 'psu_hh_idcode' not in df.columns:
        df['psu_hh_idcode'] = df['psu'].astype(str) + "_" + df['hh'].astype(str) + "_" + df['idcode'].astype(str)
        df.drop(columns=['psu', 'hh', 'idcode'], inplace=True)  # Remove individual columns after creating psu_hh_idcode

education_train['source'] = 'train'
education_test['source'] = 'test'
household_train['source'] = 'train'
household_test['source'] = 'test'

education_combined = pd.concat([education_train, education_test], axis=0).reset_index(drop=True)
household_combined = pd.concat([household_train, household_test], axis=0).reset_index(drop=True)

education_combined = education_combined.rename(columns={col: col + '_edu' for col in education_train.columns if col != 'psu_hh_idcode'})
household_combined = household_combined.rename(columns={col: col + '_hh' for col in household_train.columns if col != 'psu_hh_idcode' and col != 'hhid'})

merged_data = pd.merge(education_combined, household_combined, on='psu_hh_idcode', how='outer')
merged_data = pd.merge(merged_data, subjective_poverty_train, on='psu_hh_idcode', how='outer')

col_names = {"q06_hh": "marital_status", 
             "q07_hh": "spouse_live", 
             "q08_hh": "spouse_id", 
             "q11_hh": "mother_live", 
             "q12_hh": "mother_id", 
             "q13_hh": "mother_education", 
             "q18_hh": "father_id", 
             "q19_hh": "father_education",
             'q03_edu': "has_attended_school", 
             'q04_edu': "highest_grade_completed1",
             'q05_edu': "highest_grade_completed2",
             'q06_edu': "highest_diploma",
             'q07_edu': "no_preschool_years",
             'Q10_edu': "reason_not_going_school",
             'q01_edu': "can_read",
             'q02_edu': "can_write",
             'Q30_edu': "transport_subsidy_received",
             'Q31_edu': "transport_subsidy_received_amount",
             'Q41_edu': "spent_on_education",
             'Q47_edu': "value_textbook_subsidy",
             'Q50_edu': "private_tutoring",
             'Q53_edu': "times_private_tutor",
             'Q56_edu': "tutoring_pay",
             'Q64_edu': "scholarship",
             'Q65_edu': "scholarship_value"
             }

merged_data = merged_data.rename(columns=col_names)
merged_data[['psu', 'hh', 'idcode']] = merged_data['psu_hh_idcode'].str.split('_', expand=True).astype(int)



In [ ]:
# (merged_data).describe().to_csv("merged_data2_description.csv", index = False)

In [ ]:
from sklearn.model_selection import KFold
from lightgbm import LGBMClassifier
from sklearn.metrics import log_loss
import numpy as np

# Function for cross-validation
def cross_validate_lightgbm(X, y, y_class_labels, n_splits, reg_lambda, reg_alpha):
    cv_scores_val = []  # Validation scores
    cv_scores_train = []  # Training scores

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (train_index, val_index) in enumerate(kf.split(X), 1):
        # Split data into training and validation sets
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]

        # Get class labels for training
        y_train = y_class_labels[train_index]

        # Get one-hot encoded labels for training and validation
        y_train_onehot = y.iloc[train_index].values
        y_val_onehot = y.iloc[val_index].values

        # Train LightGBM model
        lgb_model = LGBMClassifier(
            objective='multiclass',     # Multiclass classification
            num_class=10,               # Number of classes
            random_state=42,
            reg_lambda=reg_lambda,
            reg_alpha=reg_alpha,
            metric='multi_logloss',
            learning_rate=0.01,       # Slightly increase learning rate
            n_estimators=1000,        # Increase number of estimators
            max_depth=10,     # Multiclass log loss
        )
        lgb_model.fit(X_train, y_train)

        # Predict probabilities for training and validation sets
        y_pred_proba_train = lgb_model.predict_proba(X_train)  # Training probabilities
        y_pred_proba_val = lgb_model.predict_proba(X_val)      # Validation probabilities

        # Compute multiclass log loss for training and validation
        train_loss = log_loss(y_train_onehot, y_pred_proba_train)
        val_loss = log_loss(y_val_onehot, y_pred_proba_val)

        # Append scores
        cv_scores_train.append(train_loss)
        cv_scores_val.append(val_loss)

        print(f"Fold {fold}: Training Log Loss = {train_loss:.4f}, Validation Log Loss = {val_loss:.4f}")

    avg_train_loss = np.mean(cv_scores_train)
    avg_val_loss = np.mean(cv_scores_val)

    print("\nTraining Log Loss scores for each fold:", cv_scores_train)
    print("Validation Log Loss scores for each fold:", cv_scores_val)
    print(f"Average Training Log Loss: {avg_train_loss:.4f}")
    print(f"Average Validation Log Loss: {avg_val_loss:.4f}")

    return avg_train_loss, avg_val_loss

# Grid search over reg_lambda and reg_alpha
reg_lambda_values = [1.5]#, 1.5, 2.0]
reg_alpha_values = [12.0]#, 12.0, 15.0]

best_config = None
best_val_loss = float('inf')

for reg_lambda in reg_lambda_values:
    for reg_alpha in reg_alpha_values:
        print(f"Testing reg_lambda={reg_lambda}, reg_alpha={reg_alpha}")
        avg_train_loss, avg_val_loss = cross_validate_lightgbm(X, y, y_class_labels, n_splits=7, reg_lambda=reg_lambda, reg_alpha=reg_alpha)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_config = (reg_lambda, reg_alpha)

print(f"\nBest configuration: reg_lambda={best_config[0]}, reg_alpha={best_config[1]} with Validation Log Loss: {best_val_loss:.4f}")


Testing reg_lambda=1.5, reg_alpha=12.0
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001783 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4033
[LightGBM] [Info] Number of data points in the train set: 4574, number of used features: 240
[LightGBM] [Info] Start training from score -3.240758
[LightGBM] [Info] Start training from score -2.482723
[LightGBM] [Info] Start training from score -1.814759
[LightGBM] [Info] Start training from score -1.573789
[LightGBM] [Info] Start training from score -1.586528
[LightGBM] [Info] Start training from score -1.892902
[LightGBM] [Info] Start training from score -2.362035
[LightGBM] [Info] Start training from score -3.016497
[LightGBM] [Info] Start training from score -4.872795
[LightGBM] [Info] Start training from score -6.818705
[LightGBM] [Warning] No further splits with positive gain, best gain

In [39]:
merge_test = merged_data[(merged_data['source_hh'] == 'test') | (merged_data['source_edu'] == 'test')]
psu_hh_id_merge_test = merge_test['psu_hh_idcode']
merge_test = merge_test.drop(columns=['psu_hh_idcode', 'source_hh', 'source_edu', 'hhid'])
X_test = merge_test.drop(columns=target_columns)
X_test = X_test.drop(columns=X_test.select_dtypes(include=['object']).columns)




In [ ]:
X_test.shape, X.shape

((1334, 784), (5337, 784))

In [ ]:
# Find columns in X but not in X_test
columns_in_X_not_in_X_test = set(X.columns) - set(X_test.columns)

# Find columns in X_test but not in X
columns_in_X_test_not_in_X = set(X_test.columns) - set(X.columns)

# Display the results
print("Columns in X but not in X_test:", columns_in_X_not_in_X_test)
print("Columns in X_test but not in X:", columns_in_X_test_not_in_X)


Columns in X but not in X_test: set()
Columns in X_test but not in X: set()


In [ ]:
from sklearn.model_selection import KFold
from lightgbm import LGBMClassifier
from sklearn.metrics import log_loss
import numpy as np

# Function for cross-validation

def cross_validate_lightgbm(X, y, y_class_labels, n_splits, reg_lambda, reg_alpha):
    cv_scores_val = []  # Validation scores
    cv_scores_train = []  # Training scores
    best_model = None
    best_fold_val_loss = float('inf')

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (train_index, val_index) in enumerate(kf.split(X), 1):
        # Split data into training and validation sets
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]

        # Get class labels for training
        y_train = y_class_labels[train_index]

        # Get one-hot encoded labels for training and validation
        y_train_onehot = y.iloc[train_index].values
        y_val_onehot = y.iloc[val_index].values

        # Train LightGBM model
        lgb_model = LGBMClassifier(
            objective='multiclass',     # Multiclass classification
            num_class=10,               # Number of classes
            random_state=42,
            reg_lambda=reg_lambda,
            reg_alpha=reg_alpha,
            metric='multi_logloss',
            learning_rate=0.01,       # Slightly increase learning rate
            n_estimators=1000,        # Increase number of estimators
            max_depth=10,
            verbose=-1    
        )
        lgb_model.fit(X_train, y_train)

        # Predict probabilities for training and validation sets
        y_pred_proba_train = lgb_model.predict_proba(X_train)  # Training probabilities
        y_pred_proba_val = lgb_model.predict_proba(X_val)      # Validation probabilities

        # Compute multiclass log loss for training and validation
        train_loss = log_loss(y_train_onehot, y_pred_proba_train)
        val_loss = log_loss(y_val_onehot, y_pred_proba_val)

        # Append scores
        cv_scores_train.append(train_loss)
        cv_scores_val.append(val_loss)

        print(f"Fold {fold}: Training Log Loss = {train_loss:.4f}, Validation Log Loss = {val_loss:.4f}")

        # Track the best model (lowest validation loss)
        if val_loss < best_fold_val_loss:
            best_fold_val_loss = val_loss
            best_model = lgb_model

    avg_train_loss = np.mean(cv_scores_train)
    avg_val_loss = np.mean(cv_scores_val)

    print("\nTraining Log Loss scores for each fold:", cv_scores_train)
    print("Validation Log Loss scores for each fold:", cv_scores_val)
    print(f"Average Training Log Loss: {avg_train_loss:.4f}")
    print(f"Average Validation Log Loss: {avg_val_loss:.4f}")

    return avg_train_loss, avg_val_loss, best_model

# Grid search over reg_lambda and reg_alpha
reg_lambda_values = [1.5]  # Example grid values for lambda
reg_alpha_values = [12.0]  # Example grid values for alpha

best_config = None
best_val_loss = float('inf')
best_model_overall = None

for reg_lambda in reg_lambda_values:
    for reg_alpha in reg_alpha_values:
        print(f"Testing reg_lambda={reg_lambda}, reg_alpha={reg_alpha}")
        avg_train_loss, avg_val_loss, best_model = cross_validate_lightgbm(
            X, y, y_class_labels, n_splits=7, reg_lambda=reg_lambda, reg_alpha=reg_alpha
        )

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_config = (reg_lambda, reg_alpha)
            best_model_overall = best_model

print(f"\nBest configuration: reg_lambda={best_config[0]}, reg_alpha={best_config[1]} with Validation Log Loss: {best_val_loss:.4f}")

# Use the best_model_overall for further testing or predictions.


Testing reg_lambda=1.5, reg_alpha=12.0
Fold 1: Training Log Loss = 1.6209, Validation Log Loss = 1.8934
Fold 2: Training Log Loss = 1.6028, Validation Log Loss = 1.8917
Fold 3: Training Log Loss = 1.6151, Validation Log Loss = 1.9505
Fold 4: Training Log Loss = 1.6032, Validation Log Loss = 1.9203
Fold 5: Training Log Loss = 1.6116, Validation Log Loss = 1.8735
Fold 6: Training Log Loss = 1.6045, Validation Log Loss = 1.8763
Fold 7: Training Log Loss = 1.6115, Validation Log Loss = 1.8851

Training Log Loss scores for each fold: [1.6209413973850526, 1.6027863145986798, 1.6150665246946774, 1.603188665211683, 1.6115827403597054, 1.6044934649747364, 1.6114602831137455]
Validation Log Loss scores for each fold: [1.8934180326072592, 1.8917062160393454, 1.95046199320474, 1.9203468410921967, 1.8735138850782467, 1.876300215889221, 1.8851215456285781]
Average Training Log Loss: 1.6099
Average Validation Log Loss: 1.8987

Best configuration: reg_lambda=1.5, reg_alpha=12.0 with Validation Log Los

In [37]:
from sklearn.model_selection import KFold
#from lightgbm import LGBMClassifier
from sklearn.metrics import log_loss
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Function for cross-validation

def cross_validate_rf(X, y, y_class_labels, n_splits, reg_lambda, reg_alpha):
    cv_scores_val = []  # Validation scores
    cv_scores_train = []  # Training scores
    best_model = None
    best_fold_val_loss = float('inf')

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (train_index, val_index) in enumerate(kf.split(X), 1):
        # Split data into training and validation sets
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]

        # Get class labels for training
        y_train = y_class_labels[train_index]

        # Get one-hot encoded labels for training and validation
        y_train_onehot = y.iloc[train_index].values
        y_val_onehot = y.iloc[val_index].values

        # Train LightGBM model
        # lgb_model = LGBMClassifier(
        #     objective='multiclass',     # Multiclass classification
        #     num_class=10,               # Number of classes
        #     random_state=42,
        #     reg_lambda=reg_lambda,
        #     reg_alpha=reg_alpha,
        #     metric='multi_logloss',
        #     learning_rate=0.01,       # Slightly increase learning rate
        #     n_estimators=1000,        # Increase number of estimators
        #     max_depth=10,
        #     verbose=-1    
        # )

        rf_model = RandomForestClassifier(n_estimators =75, max_depth=10)
        rf_model.fit(X_train, y_train)

        # Predict probabilities for training and validation sets
        y_pred_proba_train = rf_model.predict_proba(X_train)  # Training probabilities
        y_pred_proba_val = rf_model.predict_proba(X_val)      # Validation probabilities

        # Compute multiclass log loss for training and validation
        train_loss = log_loss(y_train_onehot, y_pred_proba_train)
        val_loss = log_loss(y_val_onehot, y_pred_proba_val)

        # Append scores
        cv_scores_train.append(train_loss)
        cv_scores_val.append(val_loss)

        print(f"Fold {fold}: Training Log Loss = {train_loss:.4f}, Validation Log Loss = {val_loss:.4f}")

        # Track the best model (lowest validation loss)
        if val_loss < best_fold_val_loss:
            best_fold_val_loss = val_loss
            best_model = rf_model

    avg_train_loss = np.mean(cv_scores_train)
    avg_val_loss = np.mean(cv_scores_val)

    print("\nTraining Log Loss scores for each fold:", cv_scores_train)
    print("Validation Log Loss scores for each fold:", cv_scores_val)
    print(f"Average Training Log Loss: {avg_train_loss:.4f}")
    print(f"Average Validation Log Loss: {avg_val_loss:.4f}")

    return avg_train_loss, avg_val_loss, best_model

# Grid search over reg_lambda and reg_alpha
reg_lambda_values = [1.5]  # Example grid values for lambda
reg_alpha_values = [12.0]  # Example grid values for alpha

best_config = None
best_val_loss = float('inf')
best_model_overall = None

for reg_lambda in reg_lambda_values:
    for reg_alpha in reg_alpha_values:
        print(f"Testing reg_lambda={reg_lambda}, reg_alpha={reg_alpha}")
        avg_train_loss, avg_val_loss, best_model = cross_validate_rf(
            X, y, y_class_labels, n_splits=7, reg_lambda=reg_lambda, reg_alpha=reg_alpha
        )

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_config = (reg_lambda, reg_alpha)
            best_model_overall = best_model

print(f"\nBest configuration: reg_lambda={best_config[0]}, reg_alpha={best_config[1]} with Validation Log Loss: {best_val_loss:.4f}")

# Use the best_model_overall for further testing or predictions.

Testing reg_lambda=1.5, reg_alpha=12.0
Fold 1: Training Log Loss = 1.6726, Validation Log Loss = 1.9444
Fold 2: Training Log Loss = 1.6870, Validation Log Loss = 1.9236
Fold 3: Training Log Loss = 1.6839, Validation Log Loss = 1.9708
Fold 4: Training Log Loss = 1.6744, Validation Log Loss = 1.9557
Fold 5: Training Log Loss = 1.6696, Validation Log Loss = 1.9158
Fold 6: Training Log Loss = 1.6673, Validation Log Loss = 1.9149
Fold 7: Training Log Loss = 1.6713, Validation Log Loss = 1.9031

Training Log Loss scores for each fold: [1.6726497218689576, 1.6869578410316783, 1.6839428175797535, 1.674357890103474, 1.6695985973771084, 1.6672912366413184, 1.6713195155067608]
Validation Log Loss scores for each fold: [1.9443700352083737, 1.9236127252221584, 1.970798772870695, 1.9556981764614148, 1.9158281401620934, 1.9148588635339556, 1.9030777593859796]
Average Training Log Loss: 1.6752
Average Validation Log Loss: 1.9326

Best configuration: reg_lambda=1.5, reg_alpha=12.0 with Validation Log L

In [21]:
from sklearn.model_selection import KFold
from sklearn import svm, datasets
import sklearn.model_selection as model_selection
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
#from lightgbm import LGBMClassifier
from sklearn.metrics import log_loss
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Function for cross-validation

def cross_validate_lightgbm(X, y, y_class_labels, n_splits, reg_lambda, reg_alpha):
    cv_scores_val = []  # Validation scores
    cv_scores_train = []  # Training scores
    best_model = None
    best_fold_val_loss = float('inf')

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (train_index, val_index) in enumerate(kf.split(X), 1):
        # Split data into training and validation sets
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]

        # Get class labels for training
        y_train = y_class_labels[train_index]

        # Get one-hot encoded labels for training and validation
        y_train_onehot = y.iloc[train_index].values
        y_val_onehot = y.iloc[val_index].values

        # Train LightGBM model
        # lgb_model = LGBMClassifier(
        #     objective='multiclass',     # Multiclass classification
        #     num_class=10,               # Number of classes
        #     random_state=42,
        #     reg_lambda=reg_lambda,
        #     reg_alpha=reg_alpha,
        #     metric='multi_logloss',
        #     learning_rate=0.01,       # Slightly increase learning rate
        #     n_estimators=1000,        # Increase number of estimators
        #     max_depth=10,
        #     verbose=-1    
        # )

        #rf_model = RandomForestClassifier()
        poly = svm.SVC(kernel='poly', degree=3, C=1).fit(X_train, y_train)


        # Predict probabilities for training and validation sets
        y_pred_proba_train = poly.predict_proba(X_train)  # Training probabilities
        y_pred_proba_val = poly.predict_proba(X_val)      # Validation probabilities

        # Compute multiclass log loss for training and validation
        train_loss = log_loss(y_train_onehot, y_pred_proba_train)
        val_loss = log_loss(y_val_onehot, y_pred_proba_val)

        # Append scores
        cv_scores_train.append(train_loss)
        cv_scores_val.append(val_loss)

        print(f"Fold {fold}: Training Log Loss = {train_loss:.4f}, Validation Log Loss = {val_loss:.4f}")

        # Track the best model (lowest validation loss)
        if val_loss < best_fold_val_loss:
            best_fold_val_loss = val_loss
            best_model = poly

    avg_train_loss = np.mean(cv_scores_train)
    avg_val_loss = np.mean(cv_scores_val)

    print("\nTraining Log Loss scores for each fold:", cv_scores_train)
    print("Validation Log Loss scores for each fold:", cv_scores_val)
    print(f"Average Training Log Loss: {avg_train_loss:.4f}")
    print(f"Average Validation Log Loss: {avg_val_loss:.4f}")

    return avg_train_loss, avg_val_loss, best_model

# Grid search over reg_lambda and reg_alpha
reg_lambda_values = [1.5]  # Example grid values for lambda
reg_alpha_values = [12.0]  # Example grid values for alpha

best_config = None
best_val_loss = float('inf')
best_model_overall = None

for reg_lambda in reg_lambda_values:
    for reg_alpha in reg_alpha_values:
        print(f"Testing reg_lambda={reg_lambda}, reg_alpha={reg_alpha}")
        avg_train_loss, avg_val_loss, best_model = cross_validate_lightgbm(
            X, y, y_class_labels, n_splits=7, reg_lambda=reg_lambda, reg_alpha=reg_alpha
        )

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_config = (reg_lambda, reg_alpha)
            best_model_overall = best_model

print(f"\nBest configuration: reg_lambda={best_config[0]}, reg_alpha={best_config[1]} with Validation Log Loss: {best_val_loss:.4f}")

# Use the best_model_overall for further testing or predictions.

Testing reg_lambda=1.5, reg_alpha=12.0


ValueError: Input X contains NaN.
SVC does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [40]:
# Ensure X_test is aligned with the training dataset (same columns)
# X_test = X_test.reindex(columns=X.columns, fill_value=0)

# Use the best model to predict probabilities on the test dataset
y_test_proba = best_model_overall.predict_proba(X_test)

# Convert predictions to the required format if necessary (e.g., for a submission file)
# Example: Assuming `target_columns` correspond to class probabilities for the test set
submission = pd.DataFrame(y_test_proba, columns=target_columns)
submission.insert(0, 'psu_hh_idcode', psu_hh_id_merge_test.values)

# Save to CSV for submission or analysis
submission.to_csv("lightgbm_submission.csv", index=False)

print("Predictions saved to 'submission.csv'.")

Predictions saved to 'submission.csv'.


In [ ]:
submission

,psu_hh_idcode,subjective_poverty_1,subjective_poverty_2,subjective_poverty_3,subjective_poverty_4,subjective_poverty_5,subjective_poverty_6,subjective_poverty_7,subjective_poverty_8,subjective_poverty_9,subjective_poverty_10
0,648_6_4,0.058918,0.054400,0.214559,0.142539,0.196995,0.154792,0.141393,0.028587,0.006087,0.001730
1,756_3_3,0.033489,0.039384,0.145725,0.175592,0.163078,0.259018,0.148489,0.026679,0.006910,0.001636
2,164_8_3,0.031238,0.055740,0.181599,0.125301,0.202968,0.175627,0.192840,0.026805,0.006137,0.001745
3,375_4_4,0.037447,0.058537,0.203635,0.146017,0.134492,0.233604,0.123741,0.049317,0.011104,0.002107
4,403_9_1,0.041079,0.045138,0.153248,0.171274,0.231040,0.181514,0.103962,0.059773,0.010903,0.002068
...,...,...,...,...,...,...,...,...,...,...,...
1329,612_1_1,0.040380,0.077423,0.191906,0.213154,0.229580,0.108685,0.100433,0.023934,0.012393,0.002112
1330,738_1_3,0.024636,0.031406,0.083726,0.174938,0.172711,0.243961,0.148868,0.100821,0.017390,0.001544
1331,191_10_2,0.062582,0.119343,0.206287,0.226618,0.186649,0.114506,0.052778,0.023574,0.005934,0.001730
1332,475_9_5,0.010126,0.044325,0.140703,0.155050,0.171884,0.240915,0.205607,0.020820,0.009030,0.001539


In [12]:
previous_submission = pd.read_csv("submission_latest.csv")

# Ensure the rows and columns are aligned
current_submission = submission.sort_values(by="psu_hh_idcode").reset_index(drop=True)
previous_submission = previous_submission.sort_values(by="psu_hh_idcode").reset_index(drop=True)

# Exclude the 'psu_hh_idcode' column
target_columns = current_submission.columns[1:]  # All columns except 'psu_hh_idcode'

# Custom log loss calculation between two probability distributions
log_losses = []
for col in target_columns:
    current_probs = current_submission[col].values
    previous_probs = previous_submission[col].values
    
    # Add a small epsilon to avoid log(0) errors
    epsilon = 1e-15
    current_probs = np.clip(current_probs, epsilon, 1 - epsilon)
    previous_probs = np.clip(previous_probs, epsilon, 1 - epsilon)
    
    # Calculate log loss for this column
    log_loss_value = -np.mean(previous_probs * np.log(current_probs))
    log_losses.append(log_loss_value)

# Average log loss across all target columns
average_log_loss = np.mean(log_losses)

print("Log Loss for each target column:", log_losses)
print("Average Log Loss between submissions:", average_log_loss)


FileNotFoundError: [Errno 2] No such file or directory: 'submission_latest.csv'

In [ ]:
log_loss(submission[target_columns].values, previous_submission[target_columns].values)

ValueError: Multioutput target data is not supported with label binarization